# 02 — Catalog Cross-Match Validation

Validate our VSCATTER-based binary/single labels against known binary catalogs: the 9th Catalogue of Spectroscopic Binary Orbits (SB9) and the Gaia DR3 Non-Single Stars (NSS) table. This provides an independent check on label quality before training.

In [ ]:
import os
import sys

import numpy as np
import matplotlib.pyplot as plt
from astropy.table import Table
from astropy.coordinates import SkyCoord
import astropy.units as u

# Add project root to path
project_root = os.path.abspath(os.path.join(os.getcwd(), "../.."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from config import LABEL_CONFIG, VALIDATION_CONFIG, VIS_CONFIG
from src.utils import crossmatch_catalogs

%matplotlib inline

## 1. Load Labeled Sample

In [ ]:
labeled = Table.read("../../data/labeled_sample.fits")
print(f"Labeled sample: {len(labeled)} stars")
print(f"  Binary (label=1): {(labeled['label'] == 1).sum()}")
print(f"  Single (label=0): {(labeled['label'] == 0).sum()}")

## 2. Query SB9 Catalog

The 9th Catalogue of Spectroscopic Binary Orbits (SB9) is a curated compilation of orbital elements for spectroscopic binaries. We query it from VizieR to check how many of our VSCATTER-labeled binaries are confirmed SB9 systems.

In [ ]:
from astroquery.vizier import Vizier

v = Vizier(columns=["*"], row_limit=-1)
sb9_tables = v.get_catalogs("B/sb9/main")
sb9 = sb9_tables[0]

print(f"SB9 catalog: {len(sb9)} entries")
print(f"Columns: {sb9.colnames}")

# Check which coordinate columns are available
has_coords = "RAJ2000" in sb9.colnames and "DEJ2000" in sb9.colnames
print(f"Has J2000 coordinates: {has_coords}")
if has_coords:
    valid_coords = ~sb9["RAJ2000"].mask & ~sb9["DEJ2000"].mask if hasattr(sb9["RAJ2000"], "mask") else np.ones(len(sb9), dtype=bool)
    print(f"Entries with valid coordinates: {valid_coords.sum()}")
    sb9_valid = sb9[valid_coords]
else:
    print("WARNING: No coordinate columns found; cross-match will not be possible.")

## 3. Cross-Match with Our Sample

In [ ]:
# Cross-match our labeled sample against SB9 at 2 arcsec
idx_ours, idx_sb9, seps = crossmatch_catalogs(
    labeled, sb9_valid,
    ra1="RA", dec1="DEC",
    ra2="RAJ2000", dec2="DEJ2000",
    radius_arcsec=2.0,
)

matched = labeled[idx_ours]
n_matched = len(matched)
n_matched_binary = (matched["label"] == 1).sum()
n_matched_single = (matched["label"] == 0).sum()

print(f"Cross-match results (2 arcsec radius):")
print(f"  Total matches with SB9: {n_matched}")
print(f"  Our binary-labeled stars in SB9: {n_matched_binary}")
print(f"  Our single-labeled stars in SB9 (missed binaries): {n_matched_single}")
print(f"  Median separation: {np.median(seps.arcsec):.3f} arcsec")

## 4. Query Gaia DR3 Non-Single Stars

In [ ]:
from astroquery.gaia import Gaia

# Query Gaia DR3 non-single star orbital solutions
job = Gaia.launch_job_async("""
SELECT source_id, ra, dec, nss_solution_type
FROM gaiadr3.nss_two_body_orbit
""")
gaia_nss = job.get_results()
print(f"Gaia DR3 NSS (two-body orbits): {len(gaia_nss)} entries")
print(f"Solution types: {np.unique(gaia_nss['nss_solution_type'])}")

# Cross-match with our labeled sample
idx_ours_gaia, idx_gaia, seps_gaia = crossmatch_catalogs(
    labeled, gaia_nss,
    ra1="RA", dec1="DEC",
    ra2="ra", dec2="dec",
    radius_arcsec=2.0,
)

matched_gaia = labeled[idx_ours_gaia]
n_gaia_match = len(matched_gaia)
n_gaia_binary = (matched_gaia["label"] == 1).sum()
n_gaia_single = (matched_gaia["label"] == 0).sum()

print(f"\nCross-match with Gaia DR3 NSS (2 arcsec):")
print(f"  Total matches: {n_gaia_match}")
print(f"  Our binary-labeled in Gaia NSS: {n_gaia_binary}")
print(f"  Our single-labeled in Gaia NSS (missed binaries): {n_gaia_single}")
print(f"  Median separation: {np.median(seps_gaia.arcsec):.3f} arcsec")

## 5. Validation Summary

In [ ]:
# Build a unified validation summary
n_total = len(labeled)
n_binary_total = (labeled["label"] == 1).sum()
n_single_total = (labeled["label"] == 0).sum()

# Combine cross-match indices (union of SB9 and Gaia matches)
known_binary_idx = np.union1d(idx_ours[matched["label"] == 1], idx_ours_gaia[matched_gaia["label"] == 1])
known_binary_in_single = np.union1d(idx_ours[matched["label"] == 0], idx_ours_gaia[matched_gaia["label"] == 0])

# Precision: of our binary-labeled stars, what fraction are confirmed?
precision_sb9 = n_matched_binary / n_binary_total if n_binary_total > 0 else 0
precision_gaia = n_gaia_binary / n_binary_total if n_binary_total > 0 else 0

# Recall: of known binaries in our footprint, what fraction did we label as binary?
n_known_in_sample_sb9 = n_matched
recall_sb9 = n_matched_binary / n_known_in_sample_sb9 if n_known_in_sample_sb9 > 0 else 0

n_known_in_sample_gaia = n_gaia_match
recall_gaia = n_gaia_binary / n_known_in_sample_gaia if n_known_in_sample_gaia > 0 else 0

print("=" * 60)
print("CROSS-MATCH VALIDATION SUMMARY")
print("=" * 60)
print(f"Our labeled sample: {n_total} stars ({n_binary_total} binary, {n_single_total} single)")
print()
print(f"{'Catalog':<20} {'Matches':>8} {'Binary':>8} {'Single':>8} {'Precision':>10} {'Recall':>8}")
print("-" * 60)
print(f"{'SB9':<20} {n_matched:>8} {n_matched_binary:>8} {n_matched_single:>8} {precision_sb9:>10.3f} {recall_sb9:>8.3f}")
print(f"{'Gaia DR3 NSS':<20} {n_gaia_match:>8} {n_gaia_binary:>8} {n_gaia_single:>8} {precision_gaia:>10.3f} {recall_gaia:>8.3f}")
print()
print("Precision = (our binaries confirmed by catalog) / (total our binaries)")
print("Recall = (our binaries confirmed by catalog) / (all catalog matches in our sample)")

# Save cross-match results
# Add SB9 and Gaia match flags to the labeled catalog
labeled["sb9_match"] = np.isin(np.arange(len(labeled)), idx_ours).astype(np.int16)
labeled["gaia_nss_match"] = np.isin(np.arange(len(labeled)), idx_ours_gaia).astype(np.int16)

out_path = "../../data/crossmatch_validation.fits"
labeled.write(out_path, overwrite=True)
print(f"\nSaved cross-match results to {out_path}")